# Full FreeSurfer `aseg.stats` task

## Goal
Run the assigned OpenNeuro T1w scan through FreeSurfer `recon-all -all`, inspect its final subcortical segmentation over the matching MRI, export every global measure and every structure row from `stats/aseg.stats`, then save the results and handover to the supervisor-shared Drive folder.

**This notebook runs FreeSurfer directly.** The full reconstruction creates `stats/aseg.stats`, including its global `# Measure` records (including eTIV) and its structure table. The notebook preserves that original file and exports all its global records and table columns.

FreeSurfer is CPU- and memory-intensive, and a full run can take hours. Use one assigned scan, keep the Colab session connected, and stop if installation or processing errors occur. The GPU is not used.

Before the task, obtain a FreeSurfer license you are authorized to use. It is free to request from the [FreeSurfer registration page](https://surfer.nmr.mgh.harvard.edu/registration.html). The notebook asks you to upload it into the temporary Colab runtime; it is not copied to Drive or saved in this notebook.

Use a Google account that has access to the shared output folder. In **Runtime → Change runtime type**, select **Runtime version 2026.07** (Ubuntu 22.04.5 LTS) and a **CPU runtime**; no GPU is needed. Then choose **File → Save a copy in Drive**. The current Colab runtime reports Ubuntu 24, while this notebook pins FreeSurfer 7.4.1 for Ubuntu 22. Confirm the OS, memory, and free disk space in the preflight cell. If the required runtime is no longer available, stop and ask your supervisor rather than bypassing the checks. Past runtime images can take a few minutes to load.

**Pilot and fallback:** This direct FreeSurfer notebook has not yet been run in Colab. Test it with a mock account before assigning it broadly. If it errors, stop and tell the supervisor; the previously tested [regional-volume notebook](https://colab.research.google.com/github/Donggyurim/neuroimaging-training/blob/bb91f9d0f2579033ee37e57b816ac56b91a33191/notebooks/First_regional_brain_volumes.ipynb) remains available as a fallback. That notebook gives regional hard-label volumes but does not produce the full `aseg.stats` measures or eTIV.

This task uses the pinned **ds004144** snapshot and one assigned T1w subject. It is a learning exercise, not a diagnostic assessment or group study. Do not infer a participant's diagnosis from the dataset title.

## Setup
### 1. Connect Drive and confirm the shared folder
Sign into the Google account your supervisor shared the output folder with. In Drive, open **Shared with me**, right-click the shared folder, choose **Organize → Add shortcut**, and place the shortcut in **My Drive**. In the next cell, enter the exact shortcut name plus the task ID and assigned subject ID. The notebook stops if it cannot find that folder.

The scan and processing files stay in Colab's temporary workspace. The notebook copies the original `aseg.stats`, parsed CSVs, QC previews, selected review files, logs, provenance, environment details, and handover into the shared folder. The FreeSurfer license is uploaded only to this Colab runtime. Do not put a license file in the shared folder, notebook, or GitHub.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import datetime, hashlib, json, os, platform, re, shlex, shutil, subprocess, sys, urllib.request

# Replace these three values with the ones in your assigned task.
TASK_ID = "task-001" #@param {type:"string"}
SUBJECT_ID = "sub-002" #@param {type:"string"}
SHARED_FOLDER_NAME = "Neuroimaging-Training" #@param {type:"string"}

DATASET_ID = "ds004144"
DATASET_COMMIT = "b6e525adf7b861d5dab7e539bff19f90ce482032"

def safe_folder_name(value, label):
    value = value.strip()
    if not value or value in {".", ".."} or "/" in value or "\\" in value:
        raise ValueError(f"Enter a valid {label}; do not include slashes.")
    return value

TASK_ID = safe_folder_name(TASK_ID, "task ID")
SUBJECT_ID = safe_folder_name(SUBJECT_ID, "subject ID")
SHARED_FOLDER_NAME = safe_folder_name(SHARED_FOLDER_NAME, "shared folder name")
if not re.fullmatch(r"sub-[A-Za-z0-9]+", SUBJECT_ID):
    raise ValueError("Use the assigned subject ID in BIDS form, such as sub-002.")

SHARED_ROOT = Path('/content/drive/MyDrive') / SHARED_FOLDER_NAME
if not SHARED_ROOT.is_dir():
    raise FileNotFoundError(
        f"Shared folder '{SHARED_FOLDER_NAME}' was not found in My Drive. "
        "Add the folder shared by your supervisor as a shortcut, then check its exact name."
    )

RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
INPUT = Path('/content/openneuro-input') / DATASET_ID / SUBJECT_ID / 'anat'
RESULTS = SHARED_ROOT / 'results' / TASK_ID / SUBJECT_ID / RUN_ID
INPUT.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=False)
WORK = Path('/content/neuroimaging-pilot') / RUN_ID
WORK.mkdir(parents=True, exist_ok=True)
print('Assigned task:', TASK_ID, '| subject:', SUBJECT_ID)
print('Your results will be saved in:', RESULTS)


### 2. Check the runtime and install Python review tools
The next cell checks that the Colab runtime matches the pinned FreeSurfer package before installing anything large. It also checks memory and free disk space, then installs only the small Python packages used to preview and export results.

In [ ]:
import importlib.util
import platform

FREESURFER_VERSION = '7.4.1'

def require_supported_colab_runtime():
    os_release = platform.freedesktop_os_release()
    pretty_name = os_release.get('PRETTY_NAME', os_release.get('ID', 'unknown'))
    version_id = os_release.get('VERSION_ID', '')
    print('Colab operating system:', pretty_name)
    if os_release.get('ID') != 'ubuntu' or not version_id.startswith('22.'):
        raise RuntimeError(
            'This notebook pins FreeSurfer 7.4.1 for Ubuntu 22. '
            f'Colab reports {pretty_name}. In Colab choose Runtime > Change runtime type > '
            'Runtime version 2026.07 (Ubuntu 22.04.5 LTS) and a CPU runtime, reconnect, '
            'then rerun from the first cell. Do not bypass this check.'
        )

require_supported_colab_runtime()

total_memory_bytes = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES')
print(f'Total runtime memory: {total_memory_bytes / (1024**3):.1f} GiB')
if total_memory_bytes < 8 * 1024**3:
    raise RuntimeError('FreeSurfer 7.4.1 needs at least 8 GiB of RAM. Stop and ask your supervisor for a larger runtime.')

minimum_free_bytes = 25 * 1024**3
free_bytes = shutil.disk_usage('/content').free
print(f'Free Colab disk space: {free_bytes / (1024**3):.1f} GiB')
if free_bytes < minimum_free_bytes:
    raise RuntimeError('At least 25 GiB of free Colab disk space is required. Stop before installing FreeSurfer.')

required_packages = {
    'numpy': 'numpy',
    'nibabel': 'nibabel',
    'pandas': 'pandas',
    'matplotlib': 'matplotlib',
    'ipywidgets': 'ipywidgets',
}
missing_packages = [
    package_name for import_name, package_name in required_packages.items()
    if importlib.util.find_spec(import_name) is None
]
if missing_packages:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--quiet', *missing_packages],
        check=True
    )

import numpy as np
import nibabel as nib
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets

print('FreeSurfer version to install:', FREESURFER_VERSION)
(RESULTS / 'environment.txt').write_text(
    subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
)

## Steps
### 3. Download the assigned scan from OpenNeuro
This notebook supports the pinned ds004144 snapshot and a subject with a file at `<subject>/anat/<subject>_T1w.nii.gz`. The notebook reads the expected byte count and MD5 from the pinned dataset's DataLad annex pointer, downloads from OpenNeuro's public S3 mirror, and verifies the bytes. If the file is absent or differs, stop and report the error; do not bypass the check.

This is one selected T1w scan, not a complete BIDS download. Inspect the printed licence, title, and scanner strength.

In [ ]:
RELATIVE_PATH = f"{SUBJECT_ID}/anat/{SUBJECT_ID}_T1w.nii.gz"
SIDECAR_RELATIVE_PATH = RELATIVE_PATH[:-7] + ".json"
IMAGE_URL = f"https://s3.amazonaws.com/openneuro.org/{DATASET_ID}/{RELATIVE_PATH}"
T1 = INPUT / f"{SUBJECT_ID}_T1w.nii.gz"

def digest(path, algorithm='sha256'):
    h = hashlib.new(algorithm)
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

annex_url = (
    f"https://api.github.com/repos/OpenNeuroDatasets/{DATASET_ID}/contents/"
    f"{RELATIVE_PATH}?ref={DATASET_COMMIT}"
)
annex_request = urllib.request.Request(
    annex_url,
    headers={
        "Accept": "application/vnd.github+json",
        "User-Agent": "neuroimaging-training-colab"
    }
)
try:
    with urllib.request.urlopen(annex_request, timeout=30) as response:
        annex_entry = json.load(response)
except Exception as exc:
    raise RuntimeError(
        "Could not look up the assigned scan in the pinned dataset snapshot. "
        "Check the subject ID and report this error to your supervisor."
    ) from exc

annex_target = annex_entry.get("target", "")
annex_filename = annex_target.rsplit("/", 1)[-1]
annex_match = re.search(r"MD5E-s(\d+)--([0-9a-fA-F]{32})", annex_filename)
if not annex_match:
    raise ValueError(
        "The assigned scan has no readable MD5 annex key in the pinned snapshot. "
        "Stop and ask your supervisor to check the task."
    )
EXPECTED_BYTES = int(annex_match.group(1))
EXPECTED_MD5 = annex_match.group(2).lower()

if not T1.exists():
    temporary = T1.with_name(T1.name + '.partial')
    with urllib.request.urlopen(IMAGE_URL, timeout=120) as response, temporary.open('wb') as handle:
        shutil.copyfileobj(response, handle)
    if temporary.stat().st_size != EXPECTED_BYTES or digest(temporary, 'md5') != EXPECTED_MD5:
        temporary.unlink(missing_ok=True)
        raise ValueError('Download differs from the pinned source. Stop and report this cell; do not bypass the check.')
    temporary.rename(T1)
assert T1.stat().st_size == EXPECTED_BYTES and digest(T1, 'md5') == EXPECTED_MD5, 'Existing input failed verification.'

metadata_root = f'https://raw.githubusercontent.com/OpenNeuroDatasets/{DATASET_ID}/{DATASET_COMMIT}/'
for relative, destination in [
    ('dataset_description.json', INPUT.parents[1] / 'dataset_description.json'),
    (SIDECAR_RELATIVE_PATH, INPUT / f'{SUBJECT_ID}_T1w.json')
]:
    with urllib.request.urlopen(metadata_root + relative, timeout=60) as response:
        payload = response.read()
    json.loads(payload)
    destination.write_bytes(payload)
description = json.loads((INPUT.parents[1] / 'dataset_description.json').read_text())
sidecar = json.loads((INPUT / f'{SUBJECT_ID}_T1w.json').read_text())
provenance = dict(
    task_id=TASK_ID, dataset=DATASET_ID, dataset_commit=DATASET_COMMIT,
    subject_id=SUBJECT_ID, dataset_doi=description.get('DatasetDOI'),
    licence=description.get('License'), source_url=IMAGE_URL,
    relative_path=RELATIVE_PATH, input_sha256=digest(T1),
    input_md5=digest(T1, 'md5'), input_bytes=T1.stat().st_size,
    freesurfer_version=FREESURFER_VERSION, run_id=RUN_ID,
    python=platform.python_version(),
    qc_status='pending', method='FreeSurfer recon-all -all; all global measures and structure rows read from stats/aseg.stats'
)
(RESULTS / 'provenance.json').write_text(json.dumps(provenance, indent=2))
print(description['Name'], '\nLicence:', description.get('License'))
print('DOI:', description.get('DatasetDOI'), '\nField strength:', sidecar.get('MagneticFieldStrength'), 'T')
print('Verified input:', T1)


### 4. Look at the MRI before processing
Check that you see a full brain, plausible contrast, and no major motion. These three slices are an initial view, not complete QC. If the scan looks unsuitable, stop and record why. The full surface run takes much longer than segmentation-only processing, so do not start it for an unsuitable scan.

In [ ]:
local_t1 = WORK / T1.name
shutil.copy2(T1, local_t1)
t1_image = nib.as_closest_canonical(nib.load(local_t1))
assert len(t1_image.shape) == 3, 'Expected one 3D anatomical volume.'
t1_data = t1_image.get_fdata(dtype=np.float32)
assert np.isfinite(t1_data).all(), 'Input has non-finite intensities.'
print('Shape:', t1_image.shape, 'Voxel size (mm):', nib.affines.voxel_sizes(t1_image.affine))
def show_input(data):
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    vmax = np.percentile(data, 99)
    for axis, (ax, title) in enumerate(zip(axes, ['Sagittal', 'Coronal', 'Axial'])):
        ax.imshow(np.take(data, data.shape[axis] // 2, axis=axis).T, origin='lower', cmap='gray', vmax=vmax)
        ax.set_title(title); ax.axis('off')
    plt.tight_layout()
    return fig
fig = show_input(t1_data)
fig.savefig(RESULTS / 'input_preview.png', dpi=150)
plt.show()


### 5. Upload a FreeSurfer license
The full surface pipeline needs a FreeSurfer license. Register for a license using the [FreeSurfer registration page](https://surfer.nmr.mgh.harvard.edu/registration.html). Upload a license file you are authorized to use in the next cell. It stays in this temporary Colab runtime and is not copied to the shared folder or included in any output.

In [ ]:
from google.colab import files
uploaded_license = files.upload()
if len(uploaded_license) != 1:
    raise ValueError('Upload exactly one FreeSurfer license file.')
license_bytes = next(iter(uploaded_license.values()))
if not license_bytes.strip():
    raise ValueError('The uploaded license file is empty.')
FS_LICENSE_PATH = Path('/content/freesurfer-license.txt')
FS_LICENSE_PATH.write_bytes(license_bytes)
del uploaded_license, license_bytes
print('FreeSurfer license received in the temporary Colab runtime. It will not be copied to Drive.')

In [ ]:
# Install the official FreeSurfer 7.4.1 package for Ubuntu 22.
FS_DEB_URL = 'https://surfer.nmr.mgh.harvard.edu/pub/dist/freesurfer/7.4.1/freesurfer_ubuntu22-7.4.1_amd64.deb'
FS_DEB_MD5 = 'bfe85dd76677cfb7ca2b247b9ac6148e'
FS_DEB_PATH = WORK / 'freesurfer_ubuntu22-7.4.1_amd64.deb'
FREESURFER_HOME = Path('/usr/local/freesurfer/7.4.1')
FS_SETUP = FREESURFER_HOME / 'SetUpFreeSurfer.sh'

require_supported_colab_runtime()
if not FS_SETUP.is_file():
    md5 = hashlib.md5()
    with urllib.request.urlopen(FS_DEB_URL, timeout=120) as response, FS_DEB_PATH.open('wb') as handle:
        downloaded = 0
        while True:
            block = response.read(8 * 1024 * 1024)
            if not block:
                break
            handle.write(block)
            md5.update(block)
            downloaded += len(block)
            if downloaded % (1024**3) < 8 * 1024 * 1024:
                print(f'Downloaded {downloaded / (1024**3):.1f} GB')
    if md5.hexdigest() != FS_DEB_MD5:
        FS_DEB_PATH.unlink(missing_ok=True)
        raise ValueError('FreeSurfer package MD5 differs from the official release listing. Stop and report this cell.')
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', str(FS_DEB_PATH)], check=True)
    FS_DEB_PATH.unlink(missing_ok=True)

if not FS_SETUP.is_file():
    raise FileNotFoundError('FreeSurfer setup script is missing after installation.')
build_stamp = FREESURFER_HOME / 'build-stamp.txt'
if not build_stamp.is_file() or FREESURFER_VERSION not in build_stamp.read_text(errors='replace'):
    raise RuntimeError('Installed FreeSurfer version does not match the pinned 7.4.1 requirement.')
with (RESULTS / 'environment.txt').open('a') as env_file:
    env_file.write(
        f'\nFreeSurfer version: {FREESURFER_VERSION}\n'
        f'Colab OS: {platform.freedesktop_os_release().get("PRETTY_NAME", "unknown")}\n'
    )
print('Installed FreeSurfer:', build_stamp.read_text(errors='replace').strip())

### 6. Run FreeSurfer `recon-all`
This runs FreeSurfer's full anatomical reconstruction for the assigned T1w scan. It creates `mri/aseg.mgz` and `stats/aseg.stats`; the stats file includes global measures and subcortical structure rows. A 3T acquisition uses FreeSurfer's `-3T` option when the dataset sidecar records 3T.

The full run can take hours. Keep Colab connected. The log is streamed to the shared results folder. If processing fails, stop and report the error; do not use partial output.

In [ ]:
SUBJECT = SUBJECT_ID
SUBJECTS_DIR = WORK / 'subjects'
SUBJECTS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT = SUBJECTS_DIR / SUBJECT
SEG = OUTPUT / 'mri/aseg.mgz'
CONFORMED = OUTPUT / 'mri/orig.mgz'
if OUTPUT.exists():
    raise FileExistsError(f'Output folder already exists: {OUTPUT}. Stop rather than overwrite it.')

NUM_THREADS = max(1, min(2, os.cpu_count() or 1))
recon_all_args = [
    'recon-all',
    '-i', str(local_t1),
    '-subjid', SUBJECT,
    '-all',
    '-openmp', str(NUM_THREADS),
]
if sidecar.get('MagneticFieldStrength') == 3:
    recon_all_args.append('-3T')

# Source FreeSurfer's environment, set the subject directory and license, then run recon-all.
shell_command = (
    f"source {shlex.quote(str(FS_SETUP))} >/dev/null && "
    f"export SUBJECTS_DIR={shlex.quote(str(SUBJECTS_DIR))} && "
    f"export FS_LICENSE={shlex.quote(str(FS_LICENSE_PATH))} && "
    f"exec {shlex.join(recon_all_args)}"
)
command = ['bash', '-c', shell_command]
provenance.update(
    freesurfer_version=FREESURFER_VERSION,
    openmp_threads=NUM_THREADS,
    command=command,
)
(RESULTS / 'provenance.json').write_text(json.dumps(provenance, indent=2))
environment = dict(
    os.environ,
    FREESURFER_HOME=str(FREESURFER_HOME),
    SUBJECTS_DIR=str(SUBJECTS_DIR),
    FS_LICENSE=str(FS_LICENSE_PATH),
)
log_path = RESULTS / 'full_pipeline.log'
with log_path.open('w') as log:
    process = subprocess.Popen(
        command, cwd=WORK, env=environment,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    for line in process.stdout:
        print(line, end='')
        log.write(line)
        log.flush()
    return_code = process.wait()
if return_code:
    raise RuntimeError(f'FreeSurfer recon-all failed ({return_code}). See full_pipeline.log; do not continue.')

if not SEG.is_file():
    raise FileNotFoundError(f'Final FreeSurfer aseg segmentation is missing: {SEG}')
if not CONFORMED.is_file():
    raise FileNotFoundError(f'Conformed MRI is missing: {CONFORMED}')
if not (OUTPUT / 'scripts/recon-all.done').is_file():
    raise FileNotFoundError('FreeSurfer did not create scripts/recon-all.done. Stop rather than using incomplete results.')
if not (OUTPUT / 'stats/aseg.stats').is_file():
    raise FileNotFoundError('FreeSurfer did not create stats/aseg.stats. Stop; do not substitute another stats file.')

REVIEW_DIR = RESULTS / 'review_files'
REVIEW_DIR.mkdir(exist_ok=True)
shutil.copy2(SEG, REVIEW_DIR / SEG.name)
shutil.copy2(CONFORMED, REVIEW_DIR / CONFORMED.name)
print('Full FreeSurfer reconstruction finished.')
print('Final aseg segmentation:', SEG)
print('Stats file:', OUTPUT / 'stats/aseg.stats')
print('Shared results:', RESULTS)

## Checks
### 7. Inspect the final segmentation over the matching MRI
Use the **conformed MRI**, which shares the final segmentation's grid. Comparing the original and segmentation arrays without alignment can show a misleading overlay. The code refuses mismatched grids.

Scroll through all three planes. Check brain coverage, ventricles, deep nuclei, and cortical boundaries. A colourful output alone does not establish that segmentation is correct. Static previews are saved; sliders require a live Colab runtime.

In [ ]:
if not CONFORMED.exists():
    raise FileNotFoundError('Matching conformed MRI mri/orig.mgz missing. Stop rather than overlaying the original grid.')
seg_image = nib.as_closest_canonical(nib.load(SEG))
conformed_image = nib.as_closest_canonical(nib.load(CONFORMED))
assert seg_image.shape == conformed_image.shape and np.allclose(seg_image.affine, conformed_image.affine), 'QC grid mismatch.'
seg = np.asarray(seg_image.dataobj)
assert np.isfinite(seg).all() and np.all(seg == np.rint(seg)), 'Expected finite integer labels.'
seg = seg.astype(np.int32)
anatomy = conformed_image.get_fdata(dtype=np.float32)
assert (seg > 0).any(), 'Segmentation is empty.'
label_ids = np.unique(seg)
colour_index = np.searchsorted(label_ids, seg)

def overlay(axis=2, fraction=0.5):
    index = min(int(fraction * (seg.shape[axis] - 1)), seg.shape[axis] - 1)
    base = np.take(anatomy, index, axis=axis).T
    labels = np.take(colour_index, index, axis=axis).T
    mask = np.take(seg, index, axis=axis).T == 0
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(base, origin='lower', cmap='gray', vmax=np.percentile(anatomy, 99))
    ax.imshow(np.ma.masked_where(mask, labels), origin='lower', cmap='turbo',
              vmin=0, vmax=len(label_ids), alpha=0.35, interpolation='nearest')
    ax.set_title(f'{["Sagittal", "Coronal", "Axial"][axis]} slice {index}')
    ax.axis('off')
    return fig

for axis in range(3):
    fig = overlay(axis)
    fig.savefig(RESULTS / f'qc_plane_{axis}.png', dpi=150)
    plt.show()

import ipywidgets as widgets
@widgets.interact(
    axis=widgets.Dropdown(options=[('Sagittal', 0), ('Coronal', 1), ('Axial', 2)], value=2),
    fraction=widgets.FloatSlider(min=0, max=1, step=0.01, value=0.5),
)
def explore(axis, fraction):
    overlay(axis, fraction)
    plt.show()

### 8. Export all measures from `aseg.stats`
The notebook preserves the original `aseg.stats` file, then exports:

- Every global `# Measure` record to `aseg_stats_global_measures.csv`, including its name, abbreviation, description, value, unit, and original record.
- Every structure row and column from `aseg.stats` to `aseg_stats_structure_stats.csv`.
- A millilitre value for measures whose unit is `mm^3`; other units remain unchanged.

The file also contains eTIV, FreeSurfer's estimated total intracranial volume. It is an estimate based on registration, not a voxel count of brain tissue. These are exploratory training outputs; inspect the segmentation and ask your supervisor to review the file before interpreting results.

In [ ]:
import io

ASEG_STATS_PATH = OUTPUT / 'stats/aseg.stats'
if not ASEG_STATS_PATH.is_file():
    raise FileNotFoundError('The full pipeline finished without stats/aseg.stats. Stop and send the task ID and full_pipeline.log to your supervisor.')
RAW_ASEG_STATS_COPY = RESULTS / 'aseg.stats'
shutil.copy2(ASEG_STATS_PATH, RAW_ASEG_STATS_COPY)

raw_stats_lines = ASEG_STATS_PATH.read_text(errors='replace').splitlines()
measure_rows = []
for line in raw_stats_lines:
    if not line.startswith('# Measure '):
        continue
    raw_record = line[len('# Measure '):]
    fields = [field.strip() for field in raw_record.split(',')]
    if len(fields) < 5:
        measure_rows.append({
            'participant_id': SUBJECT, 'measure_name': None, 'abbreviation': None,
            'description': None, 'value_raw': None, 'value': None, 'unit': None,
            'parse_status': 'unparsed', 'source_record': line
        })
        continue
    measure_name = fields[0]
    abbreviation = fields[1]
    description = ', '.join(fields[2:-2])
    value_raw = fields[-2]
    unit = fields[-1]
    try:
        value = float(value_raw)
        parse_status = 'parsed'
    except ValueError:
        value = None
        parse_status = 'non_numeric_value'
    measure_rows.append({
        'participant_id': SUBJECT,
        'measure_name': measure_name,
        'abbreviation': abbreviation,
        'description': description,
        'value_raw': value_raw,
        'value': value,
        'unit': unit,
        'parse_status': parse_status,
        'source_record': line,
    })

if not measure_rows:
    raise ValueError('No # Measure records were found in aseg.stats. Keep the raw file and report this task.')
aseg_measures = pd.DataFrame(measure_rows)
aseg_measures['value_ml'] = [
    value / 1000 if pd.notna(value) and unit in ('mm^3', 'mm3', 'mm³') else np.nan
    for value, unit in zip(aseg_measures['value'], aseg_measures['unit'])
]
aseg_measures.to_csv(RESULTS / 'aseg_stats_global_measures.csv', index=False)

header_lines = [line for line in raw_stats_lines if line.startswith('# ColHeaders ')]
if not header_lines:
    raise ValueError('No # ColHeaders table was found in aseg.stats. Keep the raw file and report this task.')
columns = header_lines[-1][len('# ColHeaders '):].split()
table_lines = [line for line in raw_stats_lines if line.strip() and not line.lstrip().startswith('#')]
if not table_lines:
    raise ValueError('No structure rows were found in aseg.stats. Keep the raw file and report this task.')
aseg_structure_stats = pd.read_csv(
    io.StringIO('\n'.join(table_lines)),
    sep=r'\s+',
    names=columns,
    header=None,
    engine='python',
)
aseg_structure_stats.insert(0, 'participant_id', SUBJECT)
aseg_structure_stats.to_csv(RESULTS / 'aseg_stats_structure_stats.csv', index=False)

print('Global measures exported:', len(aseg_measures))
print('Structure rows exported:', len(aseg_structure_stats))
display(aseg_measures)
display(aseg_structure_stats)

### 9. Record your QC decision and finish
Edit the form, then run. Use **needs_review** if uncertain; supervisor review is a separate checkpoint. Do not mark a scan as passed from only three screenshots. The stats file is a preliminary measurement until QC is complete.

In [ ]:
QC_STATUS = "needs_review" #@param ["needs_review", "pass", "fail"]
QC_NOTES = "" #@param {type:"string"}
if QC_STATUS in ('pass', 'fail') and not QC_NOTES.strip():
    raise ValueError('Describe what you inspected and why you chose pass/fail.')

provenance.update(
    qc_status=QC_STATUS,
    qc_notes=QC_NOTES,
    freesurfer_version=FREESURFER_VERSION,
    aseg_stats_sha256=digest(ASEG_STATS_PATH),
    copied_aseg_stats_sha256=digest(RAW_ASEG_STATS_COPY),
    global_measures_csv_sha256=digest(RESULTS / 'aseg_stats_global_measures.csv'),
    structure_stats_csv_sha256=digest(RESULTS / 'aseg_stats_structure_stats.csv'),
    final_segmentation_sha256=digest(SEG),
    completed_utc=datetime.datetime.now(datetime.timezone.utc).isoformat(),
)
(RESULTS / 'provenance.json').write_text(json.dumps(provenance, indent=2))

summary = f"""# Full FreeSurfer handover
Task: {TASK_ID}
Run: {RUN_ID}
Dataset: {DATASET_ID}@{DATASET_COMMIT}
Subject: {SUBJECT_ID}; input: {RELATIVE_PATH}
FreeSurfer: {FREESURFER_VERSION}
QC: {QC_STATUS}
Notes: {QC_NOTES}
Source statistics: aseg.stats
Global measure records exported: {len(aseg_measures)}
Structure rows exported: {len(aseg_structure_stats)}
Outputs: aseg.stats, aseg_stats_global_measures.csv, aseg_stats_structure_stats.csv, QC previews, provenance, environment, and full_pipeline.log.
eTIV is a registration-based estimate of intracranial volume, not brain tissue volume.
This single scan cannot establish disease differences.
Next: ask the supervisor to review the QC images and stats before interpreting any value.
"""
(RESULTS / 'handover.md').write_text(summary)
print('Saved to shared folder:', RESULTS)
print('Files:', ', '.join(sorted(p.name for p in RESULTS.iterdir())))

## Next steps
1. Open the supervisor-shared Drive folder and go to `results/<task-id>/<subject-id>/<run-id>`. Confirm `aseg.stats`, `aseg_stats_global_measures.csv`, `aseg_stats_structure_stats.csv`, QC previews, review volumes, log, provenance, environment, and handover are there.
2. Save your executed notebook copy in your own Drive. The shared results folder contains the files your supervisor needs to review.
3. Comment on your assigned task with the task ID, run ID, QC status, and any obstacle. Do not post private Drive links or personal information in a public task issue.
4. Ask your supervisor to review the QC images and the full stats output before interpreting or comparing any measure.

### Troubleshooting
- **Shared folder not found:** check the Google account and the exact shortcut name under My Drive.
- **Drive connection fails:** check that Colab is signed in to the account your supervisor shared with, then rerun cell 1.
- **Unsupported Colab OS:** choose Runtime → Change runtime type → Runtime version 2026.07 (Ubuntu 22.04.5 LTS) and a CPU runtime, reconnect, and rerun from the first cell. The GPU is not used. Do not bypass the preflight check. If that runtime is unavailable, stop and contact your supervisor.
- **Insufficient memory or disk:** use a runtime with at least 8 GiB RAM and 25 GiB free disk before installing FreeSurfer; otherwise stop and tell your supervisor.
- **FreeSurfer license error:** check that your license file is valid and uploaded to the current Colab session. Do not place it in the shared Drive or GitHub.
- **Checksum/download error:** stop and report the exact cell. Do not disable verification.
- **Import/package error:** save the message and environment details; Colab packages change.
- **Full pipeline error:** read `full_pipeline.log`; do not interpret partial outputs. FreeSurfer's subject directory stays in Colab's temporary storage.
- **Missing `stats/aseg.stats`:** do not substitute regional voxel counts or another stats file. Send the task ID and log to your supervisor.
- **Disconnected before the run completed:** runtime files may be lost; rerun the task. The full pipeline log is streamed to Drive.

### Sources and method
- [Dataset metadata at the frozen commit](https://github.com/OpenNeuroDatasets/ds004144/blob/b6e525adf7b861d5dab7e539bff19f90ce482032/dataset_description.json).
- [FreeSurfer 7.4.1 download and checksums](https://surfer.nmr.mgh.harvard.edu/fswiki/rel7downloads): source for the Ubuntu 22 package and MD5 verification.
- [FreeSurfer `recon-all`](https://surfer.nmr.mgh.harvard.edu/fswiki/recon-all): documents the full anatomical reconstruction command.
- [Google Colab past runtime versions](https://research.google.com/colaboratory/runtime-version-faq.html): runtime versions and their operating systems.
- [FreeSurfer 7.4.1 download and checksums](https://surfer.nmr.mgh.harvard.edu/fswiki/rel7downloads): source for the Ubuntu 22 package and MD5 verification.
- [FreeSurfer license registration](https://surfer.nmr.mgh.harvard.edu/registration.html).
- [FreeSurfer volume measures](https://surfer.nmr.mgh.harvard.edu/fswiki/MorphometryStats) and [eTIV method](https://surfer.nmr.mgh.harvard.edu/fswiki/eTIV).
- [NiBabel spatial images](https://nipy.org/nibabel/coordinate_systems.html): affine geometry used for QC overlays.

This notebook runs the assigned public OpenNeuro scan with FreeSurfer 7.4.1 `recon-all -all`, exports all global `# Measure` records and structure rows from FreeSurfer's `aseg.stats`, and leaves scientific interpretation and QC approval to the supervisor.